# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeda-ujala-haider/FlyRANK-Machine-Learning-First-Assignment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
method = """
MY LANE: Refresh/Content Opportunity Scoring

WHICH MODEL: RANDOM FOREST (primary) + LOGISTIC REGRESSION (comparison)

WHY RANDOM FOREST:
──────────────────
1. My signals are validated (Week ML-06)
   → Non-linear combinations matter

2. Features are clean (Week 3)
   → Won't overfit on garbage

3. Need to explain rankings (production)
   → RF permutation importance tells us which signals matter

4. Baseline is linear scoring
   → RF will show if interactions improve accuracy

WHY LOGISTIC REGRESSION (secondary):
─────────────────────────────────────
1. Simple baseline ML model
2. Fast training + interpretable
3. If LogReg beats Random Forest? Then linear was good enough
4. Compare: Does complexity help?

SUCCESS METRIC:
───────────────
Precision@50: Of top 50 articles ranked by model, how many are actually in top 50?
Precision@20: Of top 20 articles, how many are actually in top 20?

WIN CONDITION:
──────────────
Model beats Week 4 baseline on BOTH metrics
If wins @50 but loses @20: Report both (shows model strength/weakness)
"""

print(method)


MY LANE: Refresh/Content Opportunity Scoring

WHICH MODEL: RANDOM FOREST (primary) + LOGISTIC REGRESSION (comparison)

WHY RANDOM FOREST:
──────────────────
1. My signals are validated (Week ML-06)
   → Non-linear combinations matter
   
2. Features are clean (Week 3)
   → Won't overfit on garbage
   
3. Need to explain rankings (production)
   → RF permutation importance tells us which signals matter
   
4. Baseline is linear scoring
   → RF will show if interactions improve accuracy

WHY LOGISTIC REGRESSION (secondary):
─────────────────────────────────────
1. Simple baseline ML model
2. Fast training + interpretable
3. If LogReg beats Random Forest? Then linear was good enough
4. Compare: Does complexity help?

SUCCESS METRIC:
───────────────
Precision@50: Of top 50 articles ranked by model, how many are actually in top 50?
Precision@20: Of top 20 articles, how many are actually in top 20?

WIN CONDITION:
──────────────
Model beats Week 4 baseline on BOTH metrics
If wins @50 but lo

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
print("="*80)
print("SECTION 2: SPLIT DESIGN")
print("="*80)

split_design = """
CHALLENGE:
My baseline ranked ALL articles (439K rows)
But I manually reviewed only TOP 20
→ I have labels for top 20 only

HONEST SPLIT DESIGN:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Option A: TIME-AWARE SPLIT (Preferred)
─────────────────────────────────────
Use June data:
- Train: June 1-20 (20 days)
- Test: June 21-30 (10 days)

Why:
✅ Time-aware (June data sequentially splits)
✅ Realistic (train on past, test on future)
✅ Same grain as baseline
✅ No data leakage

Option B: GROUPED SPLIT (Also Good)
────────────────────────────────────
Group by client:
- Train: 80% clients (random)
- Test: 20% clients (holdout)

Why:
✅ Grouped (articles from same client in one set)
✅ Realistic (unknown client appears in future)
✅ Prevents overfitting to specific clients

OUR CHOICE: TIME-AWARE (June 1-20 vs June 21-30)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Why this split is honest:
1. Train on past → Test on future (realistic)
2. Both have full feature set (no missing data)
3. Baseline also uses June data (fair comparison)
4. Can measure ranking quality (Precision@50)
"""

print(split_design)

print("\n" + "="*80)
print("IMPLEMENTING THE SPLIT")
print("="*80)

# Code comes next

SECTION 2: SPLIT DESIGN

CHALLENGE:
My baseline ranked ALL articles (439K rows)
But I manually reviewed only TOP 20
→ I have labels for top 20 only

HONEST SPLIT DESIGN:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Option A: TIME-AWARE SPLIT (Preferred)
─────────────────────────────────────
Use June data:
- Train: June 1-20 (20 days)
- Test: June 21-30 (10 days)

Why:
✅ Time-aware (June data sequentially splits)
✅ Realistic (train on past, test on future)
✅ Same grain as baseline
✅ No data leakage

Option B: GROUPED SPLIT (Also Good)
────────────────────────────────────
Group by client:
- Train: 80% clients (random)
- Test: 20% clients (holdout)

Why:
✅ Grouped (articles from same client in one set)
✅ Realistic (unknown client appears in future)
✅ Prevents overfitting to specific clients

OUR CHOICE: TIME-AWARE (June 1-20 vs June 21-30)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Why this split is honest:
1. Train on past → Test on future (realisti

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, ndcg_score
import warnings
warnings.filterwarnings('ignore')

print("\nLOADING DATA...")

# Reload clean data from Week 4
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

dataset = load_dataset("FlyRank/internship-warehouse",
                       data_files="fact_content_daily_performance_sample.parquet")
df = dataset['train'].to_pandas()

# Filter same as Week 4
df_june = df[df['month'] == '2026-06'].copy()
df_clean = df_june[
    (df_june['gsc_data_available'] == True) &
    (df_june['ga4_data_available'] == True) &
    (df_june['gsc_impressions'] >= 10)
].drop_duplicates()

print(f"✅ Data loaded: {len(df_clean)} rows")

# ============================================================
# RECREATE FEATURES (from Week 3)
# ============================================================

print("\nRECREATING 6 FEATURES...")

# Feature 1: CTR Gap
position_ctr_benchmark = {
    1: 0.32, 2: 0.26, 3: 0.20, 4: 0.15, 5: 0.12,
    6: 0.10, 7: 0.08, 8: 0.07, 10: 0.05
}

def get_expected_ctr(position):
    position_int = int(position)
    if position_int <= 1:
        return 0.32
    elif position_int >= 10:
        return 0.05
    else:
        return position_ctr_benchmark.get(position_int, 0.10)

df_clean['ctr_expected'] = df_clean['gsc_avg_position'].apply(get_expected_ctr)
df_clean['ctr_actual'] = df_clean['gsc_clicks'] / (df_clean['gsc_impressions'] + 1)
df_clean['ctr_gap'] = df_clean['ctr_expected'] - df_clean['ctr_actual']

# Feature 2-3: Engagement metrics
df_clean['engagement_rate'] = df_clean['ga4_engaged_sessions'] / (df_clean['ga4_sessions'] + 1)
df_clean['time_on_page_sec'] = df_clean['ga4_total_engagement_sec'] / (df_clean['ga4_sessions'] + 1)

# Feature 4: Log impressions
df_clean['log_impressions'] = np.log1p(df_clean['gsc_impressions'])

# Feature 5: AI traffic percentage
df_clean['ai_traffic_pct'] = (df_clean['sessions_ai'] / (df_clean['sessions_organic'] + df_clean['sessions_direct'] + 1)) * 100
df_clean['ai_traffic_pct'] = df_clean['ai_traffic_pct'].clip(0, 100)

# Feature 6: Position tier (one-hot encoding)
df_clean['position_tier'] = pd.cut(
    df_clean['gsc_avg_position'],
    bins=[0, 3, 6, 10, 20, 1000],
    labels=['tier_top3', 'tier_top6', 'tier_top10', 'tier_top20', 'tier_below20']
)

print("✅ 6 features recreated")

# ============================================================
# CREATE TARGET (Baseline ranking as labels)
# ============================================================

print("\nCREATING TARGET FROM BASELINE...")

# We'll use: articles in top 50 = 1, rest = 0
# (Binary classification: "Is this in refresh queue?")

df_clean['day_of_month'] = pd.to_datetime(df_clean['report_date']).dt.day

# Score calculation (same as Week 4 baseline)
log_imp = np.log1p(df_clean['gsc_impressions'])
df_clean['baseline_score'] = log_imp * df_clean['ctr_gap']

# Get top 50 overall (for reference)
top_50_ids = df_clean.nlargest(50, 'baseline_score')['content_hash_id'].unique()
df_clean['is_top_50'] = df_clean['content_hash_id'].isin(top_50_ids).astype(int)

print(f"✅ Target created: {df_clean['is_top_50'].sum()} articles in top 50")

# ============================================================
# TIME-AWARE SPLIT
# ============================================================

print("\nTIME-AWARE SPLIT (June 1-20 vs June 21-30)...")

train_data = df_clean[df_clean['day_of_month'] <= 20].copy()
test_data = df_clean[df_clean['day_of_month'] > 20].copy()

print(f"Train: {len(train_data)} rows (June 1-20)")
print(f"Test: {len(test_data)} rows (June 21-30)")
print(f"Train top-50 ratio: {train_data['is_top_50'].mean():.2%}")
print(f"Test top-50 ratio: {test_data['is_top_50'].mean():.2%}")

# ============================================================
# FEATURE MATRIX
# ============================================================

print("\nPREPARING FEATURE MATRIX...")

feature_cols = [
    'ctr_gap',
    'engagement_rate',
    'time_on_page_sec',
    'log_impressions',
    'ai_traffic_pct'
]

# REORDER THIS PART

print("\nPREPARING FEATURE MATRIX...")

# ONE-HOT ENCODE FIRST (before splitting)
position_dummies = pd.get_dummies(df_clean['position_tier'], prefix='position')
df_clean = pd.concat([df_clean, position_dummies], axis=1)

print("✅ Position tier one-hot encoded")

# THEN CREATE TRAIN/TEST SPLIT
print("\nTIME-AWARE SPLIT (June 1-20 vs June 21-30)...")

train_data = df_clean[df_clean['day_of_month'] <= 20].copy()
test_data = df_clean[df_clean['day_of_month'] > 20].copy()

print(f"Train: {len(train_data)} rows (June 1-20)")
print(f"Test: {len(test_data)} rows (June 21-30)")
print(f"Train top-50 ratio: {train_data['is_top_50'].mean():.2%}")
print(f"Test top-50 ratio: {test_data['is_top_50'].mean():.2%}")

# NOW ACCESS FEATURES
feature_cols = [
    'ctr_gap',
    'engagement_rate',
    'time_on_page_sec',
    'log_impressions',
    'ai_traffic_pct'
]

position_cols = [col for col in position_dummies.columns]
all_features = feature_cols + position_cols

print(f"\nTotal features: {len(all_features)}")
print(f"Features: {all_features}")

# Handle missing values
df_clean[all_features] = df_clean[all_features].fillna(0)

# Split features and target
X_train = train_data[all_features].copy()
y_train = train_data['is_top_50'].copy()

X_test = test_data[all_features].copy()
y_test = test_data['is_top_50'].copy()

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ SPLIT READY")
print(f"X_train shape: {X_train_scaled.shape}")
print(f"X_test shape: {X_test_scaled.shape}")
print(f"y_train distribution: {y_train.value_counts().to_dict()}")
print(f"y_test distribution: {y_test.value_counts().to_dict()}")


LOADING DATA...


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.